# Reading Accumulation Results: Connections, Misses, and Self-Connections

The Python port of the R `tma` package's accumulation-results vignette.

`accumulate()` returns more than one network per unit. Alongside the familiar
**`connection_counts_`**, it returns **`miss_counts_`** (missed
co-occurrences) and **`self_connection_counts_`** / **`self_miss_counts_`**
(each code's connection to itself). Each comes in two forms: a unit-level
array, and a line-level `row_*` array.

This notebook explains what each array counts, how it is laid out, and how the
arrays relate to one another. It uses the same sample data, horizons and
context tensor as [Getting Started with TMA in
Python](getting-started.ipynb), and needs `qe-tma` 0.1.11 or later.

In [1]:
import pandas as pd
from tma import accumulate, context_tensor, rules
from tma.datasets import load_test_mockdata

test_mockdata = load_test_mockdata()
sample_df = test_mockdata[test_mockdata["chatGroup"] == "PAM"].reset_index(drop=True)

unit_cols = ["userID", "condition"]
codes = ["A", "B", "C"]

hoo_rules = rules(
    lambda df, unit: (df["modality"] == "chat")
                     & (df["chatGroup"] == unit["chatGroup"])
                     & (df["condition"] == unit["condition"]),
    lambda df, unit: (df["modality"] == "resource")
                     & (df["userID"] == unit["userID"])
                     & (df["condition"] == unit["condition"]),
)

tensor = context_tensor(
    sample_df,
    sender_cols=None,
    receiver_cols=None,
    mode_col="modality",
    default_window=0,
    default_weight=0,
)
chat = tensor.mode_levels.index("chat")
resource = tensor.mode_levels.index("resource")
WEIGHT, WINDOW = 0, 1
tensor[chat, WINDOW] = 360
tensor[resource, WINDOW] = 180
tensor[chat, WEIGHT] = 1.0
tensor[resource, WEIGHT] = 2.0

accum = accumulate(
    sample_df,
    units=unit_cols,
    codes=codes,
    hoo_rules=hoo_rules,
    tensor=tensor,
    time_col="timeStamp",
    ordered=True,
    weight_by="product",
)
accum

<TMAAccumulation 3 units x 9 connections (ordered)>

We use an ordered (directed) model with the `"product"` weight model so that
the numbers below are the raw, unweighted counts; [Weight
models](#Weight-models) covers the other options.

Two small helpers turn the arrays into labelled tables: one row per unit, or
one row per data line.

In [2]:
def by_unit(values, columns):
    return pd.DataFrame(values, index=accum.unit_labels_, columns=columns)


def by_line(values, columns):
    lines = pd.MultiIndex.from_arrays(
        [range(1, len(sample_df) + 1), sample_df["userID"]], names=["line", "userID"]
    )
    return pd.DataFrame(values, index=lines, columns=columns)


cols = accum.connection_names_

---

## What is returned

| Unit level (one row per unit) | Line level (one row per line) | R | Counts |
|---|---|---|---|
| `connection_counts_` | `row_connection_counts_` | `connection.counts` | Ground code *i* → response code *j*, both present |
| `miss_counts_` | `row_miss_counts_` | `miss.counts` | Ground code *i* present, response code *j* **absent** |
| `self_connection_counts_` | `row_self_connection_counts_` | `self.connection.counts` | Ground code *i* → response code *i* |
| `self_miss_counts_` | `row_self_miss_counts_` | `self.miss.counts` | Ground code *i* present, response code *i* **absent** |

The miss arrays are `None` unless `misses=True` (the default). Set
`misses=False` to skip them, which saves memory with many codes or long data.

Line-level arrays always hold the **raw** counts for each line, one row per
row of the data. Unit-level arrays apply the weight model to each line and then
sum the lines for each unit.

---

## Lines, grounds, and responses

Every row of a unit (every **line**) is accumulated as a **response**, $r$.
Its **ground**, $g$, is the weighted sum of the codes on the earlier rows in
the unit's horizon that are still inside their modality's window:

$$
g = \sum_{\text{in-window rows } k} W_{m(k)} \, c_k
$$

where $c_k$ is the code vector of row $k$ and $W_{m(k)}$ is the weight of its
modality. Everything below is computed from $g$ and $r$ for one line at a time.

Here are the rows of our sample, with the codes each one carries (numbered
from 1, to match the line numbers used below):

In [3]:
sample_df[["userID", "modality", "timeStamp", "A", "B", "C"]].set_axis(range(1, len(sample_df) + 1))

,userID,modality,timeStamp,A,B,C
1,User1,chat,1379410965,1,0,0
2,User2,resource,1379411071,0,1,1
3,User3,chat,1379411300,1,0,0
4,User1,resource,1379411494,0,0,1
5,User2,chat,1379411667,0,1,0
6,User3,chat,1379411813,1,0,1
7,User1,resource,1379500979,1,0,0
8,User2,resource,1379501110,0,1,1
9,User3,chat,1379501265,1,0,0
10,User1,resource,1379501471,0,0,1


---

## `connection_counts_`: connections made

For one line, the directed connection from code $i$ to code $j$ is

$$
\text{conn}_{i \to j} = g_i \, r_j \;+\; \tfrac{1}{2} \, W_r \, r_i \, r_j \quad (i \neq j)
$$

* **$g_i \, r_j$** — code $i$ in the ground, code $j$ in the response.
* **$\tfrac{1}{2} W_r \, r_i \, r_j$** — codes $i$ and $j$ appearing together
  *in the response itself*, weighted by the response's own modality weight
  $W_r$. Half goes to $i \to j$ and half to $j \to i$, so the pair counts once
  in total.

Columns are named **`"ground & response"`**: `"B & C"` is B in the ground
connecting to C in the response.

In [4]:
by_line(accum.row_connection_counts_, cols)

,,A & A,B & A,C & A,A & B,B & B,C & B,A & C,B & C,C & C
line,userID,,,,,,,,,
1,User1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,User2,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0
3,User3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,User1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5,User2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,User3,0.0,1.0,0.5,0.0,0.0,0.0,0.5,1.0,0.0
7,User1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,User2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
9,User3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Three lines worked by hand:

* **Line 2** (User2 resource click, `B, C`). The ground is line 1 (User1 chat,
  `A`, 106 s earlier, inside the 360 s chat window), so `A & B` = `A & C` = 1.
  B and C also co-occur in the response, a resource click with weight 2, so
  `B & C` and `C & B` each get ½ × 2 = 1.
* **Line 4** (User1 resource click, `C`). Line 3 (User3 chat, `A`, 194 s
  earlier) is in the ground; line 1 is 529 s back and has expired. So
  `A & C` = 1.
* **Line 6** (User3 chat, `A, C`). The ground is line 5 (User2 chat, `B`), so
  `B & A` = `B & C` = 1. A and C co-occur in the chat response (weight 1), so
  `A & C` and `C & A` each get ½.

The unit-level array is the sum of each unit's lines, after the weight model:

In [5]:
by_unit(accum.connection_counts_, cols)

,A & A,B & A,C & A,A & B,B & B,C & B,A & C,B & C,C & C
User1::FirstHalf,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
User2::FirstHalf,0.0,0.0,0.0,2.0,0.0,2.0,1.0,2.0,0.0
User3::FirstHalf,1.0,2.0,1.0,0.0,0.0,0.0,1.0,2.0,0.0


### Unordered networks

With `ordered=False`, each line is folded to the upper triangle before it is
weighted and summed: the undirected `"A & B"` is $\text{conn}_{A \to B} +
\text{conn}_{B \to A}$. Self-connections (`A & A`, …) are dropped from
`connection_counts_` unless you set `self_connections=True`. See
[Self-connections](#self_connection_counts_:-a-code-following-itself).

---

## `miss_counts_`: connections not made

A miss is the other side of a connection: the ground carried code $i$, but the
response did **not** carry code $j$.

$$
\text{miss}_{i \to j} = g_i \, [\, r_j = 0 \,]
$$

There is no within-response term: a miss is always between the ground and the
response. Columns use the same `"ground & response"` names and layout as
`connection_counts_`.

In [6]:
by_line(accum.row_miss_counts_, cols)

,,A & A,B & A,C & A,A & B,B & B,C & B,A & C,B & C,C & C
line,userID,,,,,,,,,
1,User1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,User2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,User3,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,User1,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5,User2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,User3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
7,User1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,User2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,User3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


* **Line 3** (User3 chat, `A`). The ground is line 1 (`A`). The response has
  no B and no C, so `A & B` = `A & C` = 1.
* **Line 4** (User1 resource, `C`). The ground is line 3 (`A`); the response
  has no A or B, so `A & A` = `A & B` = 1. `A & C` is a connection, not a miss.
* **Line 6** (User3 chat, `A, C`). The ground is line 5 (`B`); the response
  has no B, so `B & B` = 1.

In [7]:
by_unit(accum.miss_counts_, cols)

,A & A,B & A,C & A,A & B,B & B,C & B,A & C,B & C,C & C
User1::FirstHalf,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0
User2::FirstHalf,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
User3::FirstHalf,0.0,0.0,0.0,1.0,2.0,0.0,1.0,0.0,0.0


Use misses where the absence of a connection is informative: how often a code
in the ground was *not* followed by another code, rather than only how often it
was.

### Connections and misses together

With 0/1 codes, every unit of ground weight on code $i$ is either followed by
code $j$ or not, so for each line

$$
g_i \, r_j + \text{miss}_{i \to j} = g_i .
$$

That gives the share of code $i$'s ground in which $j$ followed. The
connection also contains the within-response term, so remove that first; for
self-connections there is no such term (see below), and the identity holds as
it stands.

With unordered networks, misses are folded like connections: the undirected
`"A & B"` miss is $\text{miss}_{A \to B} + \text{miss}_{B \to A}$.

---

## `self_connection_counts_`: a code following itself

The self-connection of code $i$ is the diagonal of the directed network:

$$
\text{self}_i = g_i \, r_i
$$

Code $i$ is in the ground and in the response: the code persists or recurs.
There is no within-response term (a code does not co-occur with itself).

* Ordered networks already contain it, as the `"A & A"`, `"B & B"`, `"C & C"`
  columns of `connection_counts_`.
* Unordered networks fold the diagonal away, so `connection_counts_` has no
  self columns by default.

`self_connection_counts_` keeps the self-connections either way, one column
per code. It is reported alongside the model and does not enter it.

In [8]:
by_unit(accum.self_connection_counts_, codes)

,A,B,C
User1::FirstHalf,0.0,0.0,0.0
User2::FirstHalf,0.0,0.0,0.0
User3::FirstHalf,1.0,0.0,0.0


In [9]:
by_line(accum.row_self_connection_counts_, codes)

,,A,B,C
line,userID,,,
1,User1,0.0,0.0,0.0
2,User2,0.0,0.0,0.0
3,User3,1.0,0.0,0.0
4,User1,0.0,0.0,0.0
5,User2,0.0,0.0,0.0
6,User3,0.0,0.0,0.0
7,User1,0.0,0.0,0.0
8,User2,0.0,0.0,0.0
9,User3,0.0,0.0,0.0


Line 3 (User3 chat, `A`) has line 1 (`A`) in its ground, so its
self-connection for A is 1. That is the same value as the `"A & A"` column of
`row_connection_counts_` above.

### `self_miss_counts_`

The self-miss is code $i$ in the ground and absent from the response:

$$
\text{selfmiss}_i = g_i \, [\, r_i = 0 \,]
$$

In [10]:
by_unit(accum.self_miss_counts_, codes)

,A,B,C
User1::FirstHalf,2.0,0.0,0.0
User2::FirstHalf,2.0,0.0,0.0
User3::FirstHalf,0.0,2.0,0.0


With 0/1 codes, self-connections and self-misses split the ground exactly, so
their sum is each unit's total ground weight on each code, and
`self / (self + self_miss)` is how often a code in the ground was repeated:

In [11]:
by_unit(accum.self_connection_counts_ + accum.self_miss_counts_, codes)

,A,B,C
User1::FirstHalf,2.0,0.0,0.0
User2::FirstHalf,2.0,0.0,0.0
User3::FirstHalf,1.0,2.0,0.0


### Self-connections in an unordered model

To make self-connections part of an unordered network, so that they count
toward the line weights, the rotation and the node positions, pass
`self_connections=True`. The diagonal is then added to `connection_counts_`
(and the self-misses to `miss_counts_`), in the order `A & A`, `A & B`,
`B & B`, `A & C`, …:

In [12]:
accum_self = accumulate(
    sample_df,
    units=unit_cols,
    codes=codes,
    hoo_rules=hoo_rules,
    tensor=tensor,
    time_col="timeStamp",
    ordered=False,
    self_connections=True,
)
accum_self.connection_names_

['A & A', 'A & B', 'B & B', 'A & C', 'B & C', 'C & C']

`self_connections` has no effect on ordered models, which always carry the
diagonal.

---

## Weight models

`weight_by` sets how each line's counts are transformed before the lines are
summed into a unit's network. It applies the same way to connections, misses,
self-connections and self-misses.

| `weight_by` | Each line's count becomes |
|---|---|
| `"binary"` (default with `binary=True`) | Unordered: 1 if positive, else 0. Ordered: unchanged. |
| `"product"` (default with `binary=False`) | Unchanged |
| `"sqrt"` | $\sqrt{x}$ |
| `"log1p"` (alias `"log"`) | $\log(1 + x)$ |

For unordered models the weight is applied after each line is folded to the
upper triangle; for ordered models, to each directed cell. The line-level
`row_*` arrays always hold the raw, unweighted counts, so you can re-weight
them yourself.

---

## Summary

* **`connection_counts_`** — ground code *i* followed by response code *j*,
  plus *i* and *j* together in the response.
* **`miss_counts_`** — ground code *i* *not* followed by response code *j*.
* **`self_connection_counts_`** — ground code *i* followed by itself; part of
  ordered networks, kept separately for unordered ones.
* **`self_miss_counts_`** — ground code *i* not followed by itself.
* **`row_*`** — the same quantities for every line, raw and unweighted.